In [ ]:
import numpy as np
import pandas as pd
import pickle
from sklearn.model_selection import train_test_split, KFold

In [ ]:
md = pd.read_csv("../neuron_metadata/bugeon.csv")
md

In [ ]:
test_subject = ["SB028"]
ssl_val_sessions = ["sb025_20191004", "sb026_20191011", "sb028_20191106", "sb030_20200108"]

In [ ]:
# Transductive split E/I

md = pd.read_csv("../neuron_metadata/bugeon.csv")
test_size = len(md) // 5
test_md = md[~np.isin(md.session_id, ssl_val_sessions)]  # from which we can sample test
_, test_ids = train_test_split(test_md.id.values, test_size=test_size, random_state=42, stratify=test_md.ei_class.values)

trainval_md = md[~np.isin(md.id.values, test_ids)]
train_ids, val_ids = train_test_split(trainval_md.id.values, test_size=0.2, random_state=42, stratify=trainval_md.ei_class.values)

print(len(train_ids), len(val_ids), len(test_ids))
split = {
    "train": train_ids,
    "val": val_ids,
    "test": test_ids,
}

label_map = pd.DataFrame(md.ei_class.to_numpy(), md.id.to_numpy(), columns=["label"])
data = {
    "splits": split,
    "label_map": label_map,
}

output_file = "../splits/bugeon_ei_within.pkl"
with open(output_file, "wb") as f:
    pickle.dump(data, f)
print(f"Output written to {output_file}")

In [ ]:
# Transductive split subclass

md = pd.read_csv("../neuron_metadata/bugeon.csv")
md = md[(md.subclass != "unknown") & (md.subclass != "Serpinf1")]

test_size = len(md) // 5
test_md = md[~np.isin(md.session_id, ssl_val_sessions)]  # from which we can sample test
_, test_ids = train_test_split(test_md.id.values, test_size=test_size, random_state=42, stratify=test_md.subclass.values)
test_mask = np.isin(md.id.values, test_ids)

trainval_md = md[~test_mask]
train_ids, val_ids = train_test_split(trainval_md.id.values, test_size=0.2, random_state=42, stratify=trainval_md.subclass.values)

print(len(train_ids), len(val_ids), len(test_ids))
split = {
    "train": train_ids,
    "val": val_ids,
    "test": test_ids,
}

label_map = pd.DataFrame(md.subclass.to_numpy(), md.id.to_numpy(), columns=["label"])
data = {
    "splits": split,
    "label_map": label_map,
}

output_file = "../splits/bugeon_subclass_within.pkl"
with open(output_file, "wb") as f:
    pickle.dump(data, f)
print(f"Output written to {output_file}")

In [ ]:
# Offline across-population split E/I

md = pd.read_csv("../neuron_metadata/bugeon.csv")
ssl_val_mask = np.isin(md.session_id, ssl_val_sessions)

splits = []
for test_subject in np.sort(md.subject_id.unique()):
    test_mask = (~ssl_val_mask) & (md.subject_id == test_subject)
    test_ids = md[test_mask].id.values

    cv_md = md[md.subject_id != test_subject]
    cv_splits = []
    for cv_val_subject in np.sort(cv_md.subject_id.unique()):
        val_ids = cv_md[cv_md.subject_id == cv_val_subject].id.values
        train_ids = cv_md[cv_md.subject_id != cv_val_subject].id.values
        cv_splits.append({
            "train": train_ids,
            "val": val_ids,
        })
        assert not np.isin(train_ids, test_ids).any()
        assert not np.isin(val_ids, test_ids).any()
        assert not np.isin(val_ids, train_ids).any()


        print(test_subject, len(train_ids), len(val_ids), len(test_ids))
    
    splits.append({
        "test": test_ids,
        "train_cv": cv_splits,
        "train": md[md.subject_id != test_subject].id.values,
    })

label_map = pd.DataFrame(md.ei_class.to_numpy(), md.id.to_numpy(), columns=["label"])
data = {
    "splits": splits,
    "label_map": label_map,
}

output_file = "../splits/bugeon_ei_across.pkl"
with open(output_file, "wb") as f:
    pickle.dump(data, f)
print(f"Output written to {output_file}")


In [ ]:
# Offline across-population split subclass

md = pd.read_csv("../neuron_metadata/bugeon.csv")
md = md[(md.subclass != "unknown") & (md.subclass != "Serpinf1")]
ssl_val_mask = np.isin(md.session_id, ssl_val_sessions)

splits = []
for test_subject in np.sort(md.subject_id.unique()):
    test_mask = (~ssl_val_mask) & (md.subject_id == test_subject)
    test_ids = md[test_mask].id.values

    cv_md = md[md.subject_id != test_subject]
    cv_splits = []
    for cv_val_subject in np.sort(cv_md.subject_id.unique()):
        val_ids = cv_md[cv_md.subject_id == cv_val_subject].id.values
        train_ids = cv_md[cv_md.subject_id != cv_val_subject].id.values
        cv_splits.append({
            "train": train_ids,
            "val": val_ids,
        })
        assert not np.isin(train_ids, test_ids).any()
        assert not np.isin(val_ids, test_ids).any()
        assert not np.isin(val_ids, train_ids).any()


        print(test_subject, len(train_ids), len(val_ids), len(test_ids))
    
    splits.append({
        "test": test_ids,
        "train_cv": cv_splits,
        "train": md[md.subject_id != test_subject].id.values,
    })

label_map = pd.DataFrame(md.subclass.to_numpy(), md.id.to_numpy(), columns=["label"])
data = {
    "splits": splits,
    "label_map": label_map,
}

output_file = "../splits/bugeon_subclass_across.pkl"
with open(output_file, "wb") as f:
    pickle.dump(data, f)
print(f"Output written to {output_file}")


In [ ]:
# Online across-population split (split 3) E/I

md = pd.read_csv("../neuron_metadata/bugeon.csv")
ssl_val_mask = np.isin(md.session_id, ssl_val_sessions)

splits = []
test_subject = "SB028"
test_mask = (~ssl_val_mask) & (md.subject_id == test_subject)
test_ids = md[test_mask].id.values

cv_md = md[md.subject_id != test_subject]
cv_splits = []
for cv_val_subject in cv_md.subject_id.unique():
    val_ids = md[md.subject_id == cv_val_subject].id.values
    train_ids = md[md.subject_id != cv_val_subject].id.values
    cv_splits.append({
        "train": train_ids,
        "val": val_ids,
    })
    print(test_subject, len(train_ids), len(val_ids), len(test_ids))

splits.append({
    "test": test_ids,
    "train_cv": cv_splits,
    "train": md[md.subject_id != test_subject].id.values,
})

label_map = pd.DataFrame(md.ei_class.to_numpy(), md.id.to_numpy(), columns=["label"])
data = {
    "splits": splits,
    "label_map": label_map,
}

output_file = "../splits/bugeon_ei_sb028.pkl"
with open(output_file, "wb") as f:
    pickle.dump(data, f)
print(f"Output written to {output_file}")

In [ ]:
# Offline across-population split (split3) subclass

md = pd.read_csv("../neuron_metadata/bugeon.csv")
md = md[(md.subclass != "unknown") & (md.subclass != "Serpinf1")]
ssl_val_mask = np.isin(md.session_id, ssl_val_sessions)

splits = []
test_subject = "SB028"
test_mask = (~ssl_val_mask) & (md.subject_id == test_subject)
test_ids = md[test_mask].id.values

cv_md = md[md.subject_id != test_subject]
cv_splits = []
for cv_val_subject in cv_md.subject_id.unique():
    val_ids = cv_md[cv_md.subject_id == cv_val_subject].id.values
    train_ids = cv_md[cv_md.subject_id != cv_val_subject].id.values
    print(test_subject, len(train_ids), len(val_ids), len(test_ids))
    cv_splits.append({
        "train": train_ids,
        "val": val_ids,
    })

splits.append({
    "test": test_ids,
    "train_cv": cv_splits,
    "train": md[md.subject_id != test_subject].id.values,
})

label_map = pd.DataFrame(md.subclass.to_numpy(), md.id.to_numpy(), columns=["label"])
data = {
    "splits": splits,
    "label_map": label_map,
}

output_file = "../splits/bugeon_subclass_sb028.pkl"
with open(output_file, "wb") as f:
    pickle.dump(data, f)
print(f"Output written to {output_file}")